# 02. PE label and prior audit

Start from the official lightweight summary tables, then compare them with the deep PESummary inspection reports. This notebook discovers candidates; it does not freeze a production label or reconstruct a prior.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd

inventory_dir = Path(os.environ["GWTC_RESULTS_ROOT"]) / "pilot" / "inventory"
index_path = inventory_dir / "pe_release_index.json"
index = json.loads(index_path.read_text())
index["ready_for_label_audit"], index_path

In [ ]:
candidate_rows = []
for release, payload in index["releases"].items():
    if not payload.get("present", False):
        continue
    for event_id, event in payload["events"].items():
        for row in event["rows"]:
            candidate_rows.append({"release": release, "event_id": event_id, **row})
candidates = pd.DataFrame(candidate_rows)
candidates[[
    "release", "event_id", "result_samples_key", "result_file_name",
    "mass_1_source_median", "mass_2_source_median", "redshift_median"
]] if len(candidates) else candidates

In [ ]:
deep_rows = []
for path in sorted(inventory_dir.glob("pe_*_deep.json")):
    report = json.loads(path.read_text())
    event_id = path.name.split("_")[1] if path.name.startswith("pe_GW") else path.stem
    for analysis in report.get("analyses", []):
        deep_rows.append({
            "report": path.name,
            "event_id": event_id,
            "label": analysis["label"],
            "sample_count": analysis["sample_count"],
            "parameter_count": len(analysis["parameters"]),
            "has_mass_1_source": "mass_1_source" in analysis["parameters"],
            "has_mass_2_source": "mass_2_source" in analysis["parameters"],
            "has_redshift": "redshift" in analysis["parameters"],
        })
pd.DataFrame(deep_rows)

## Required decision record

For each event, record: (1) every released candidate label, (2) the official population-analysis label/mixing policy, (3) available prior and configuration metadata, (4) sampled coordinates, (5) source-frame transformations and Jacobians, and (6) an independent numerical prior check. Only then update `config/pe_mapping.toml`. A tutorial waveform-priority example is not automatically the population-analysis policy.